# 11 하이브리드 수요예측

**type별 SBC(rule-base) vs ML(PatchTST+HAC)** — 10장과 동일한 **XGBoost+Best 임베딩** 예측으로, 제품수 반영 **가중 WMAPE**만 비교합니다.

## 논문과의 대응

| 논문 (Ecuador) | 본 실험 |
|----------------|---------|
| Center **B** (고변동) | type **B, E** 등 — CV 높은 매장 |
| Center **A** (저변동) | type **D, A** 등 — CV 상대적 낮음 |
| SBC가 고변동에서 유리 | 가중 WMAPE **SBC 3 : ML 2** |
| ML이 저변동에서 유리 | **D → ML**, **C → ML**(예외) |

> **주의:** SBC·ML이 데이터 구조마다 항상 동일하게 우세하지는 않습니다.  
> 다만 **변동성(CV)과 scheme 선택이 어느 정도 연관**된다는 점은, 본 실험에서도 부분적으로 확인됩니다 (11장 ②절).

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.experiment_data import load_forecast_frames

df, feat_df = load_forecast_frames()
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
print('Phase2 (XGBoost×6임베딩) | rows:', len(phase2), '| 조건:', len(phase2_best))


Phase2 (XGBoost×6임베딩) | rows: 1980 | 조건: 34


### ① 가중 WMAPE — type별 SBC vs ML

In [2]:
import numpy as np
from utils.phase_analysis import (
    validation_weights, build_family_from_phase2_best,
    compare_schemes_by_type,
)

val_weights = validation_weights(df)
family_final = build_family_from_phase2_best(phase2, phase2_best, val_weights)

type_compare = compare_schemes_by_type(family_final)
display(type_compare)
print('가중 WMAPE 우세:', type_compare['better_scheme_weighted'].value_counts().to_dict())

# --- type별 수요 변동성 (CV) ---
train = df[df['yearweek'] <= TRAIN_WEEK_MAX]

# (1) 제품(family)별 CV → type 평균 — SKU 수요 변동성 특성
fam_cv = train.groupby(['type', 'family'])['sales'].apply(
    lambda s: s.std() / s.mean() if s.mean() > 0 else np.nan
)
cv_family = fam_cv.groupby('type').agg(['mean', 'std', 'count']).round(3)
cv_family.columns = ['cv_mean', 'cv_std', 'n_family']
print('\n=== type별 제품(family) 평균 CV — 수요 변동성 ===')
print(cv_family)

# (2) type 내 전 주간 판매 record 합산 CV — 분포 규모 참고 (mean/std는 절대 판매량 수준)
cv_pooled = train.groupby('type')['sales'].agg(['mean', 'std'])
cv_pooled['cv'] = (cv_pooled['std'] / cv_pooled['mean']).round(3)
print('\n=== type별 학습구간 판매 record CV (참고) ===')
print(cv_pooled)

# CV 순위 vs scheme 우세 대조
rank = cv_pooled['cv'].rank(ascending=False).astype(int)
summary = type_compare[['SBC_wmape_weighted', 'ML_wmape_weighted', 'better_scheme_weighted']].copy()
summary['cv_rank'] = rank
summary['cv_pooled'] = cv_pooled['cv']
print('\n=== CV 순위 × scheme 우세 ===')
display(summary)


,n_products,SBC_wmape_weighted,ML_wmape_weighted,SBC_wmape_mean,ML_wmape_mean,delta_weighted_SBC_minus_ML,better_scheme_weighted
A,33,34.23,37.17,68.44,124.32,-2.94,SBC
B,33,28.89,26.59,84.20,54.33,2.30,ML
C,33,32.91,34.04,51.06,106.37,-1.12,SBC
D,33,37.24,38.85,73.04,60.38,-1.61,SBC
E,33,28.25,27.64,42.97,36.88,0.62,ML


가중 WMAPE 우세: {'SBC': 3, 'ML': 2}

=== type별 제품(family) 평균 CV — 수요 변동성 ===
      cv_mean  cv_std  n_family
type                           
A       0.621   0.673        32
B       0.622   0.424        32
C       0.719   0.842        32
D       0.617   0.646        33
E       0.868   0.877        32

=== type별 학습구간 판매 record CV (참고) ===
              mean            std     cv
type                                    
A     44216.910475  100342.898923  2.269
B     18162.835311   45927.000022  2.529
C     20579.720066   53018.856163  2.576
D     44015.913722   99010.064531  2.249
E      7464.302488   19194.165941  2.571

=== CV 순위 × scheme 우세 ===


,SBC_wmape_weighted,ML_wmape_weighted,better_scheme_weighted,cv_rank,cv_pooled
A,34.23,37.17,SBC,4,2.269
B,28.89,26.59,ML,3,2.529
C,32.91,34.04,SBC,1,2.576
D,37.24,38.85,SBC,5,2.249
E,28.25,27.64,ML,2,2.571


### ② 해석 — 변동성(CV)과 SBC/ML 선택

**가중 WMAPE** = Σ(WMAPE_f × 검증판매량_f) / Σ(검증판매량_f) — type 전체를 하나의 지표로 비교할 때만 사용.

#### 결과 요약
| type | 가중 WMAPE 우세 | record CV (참고) | 해석 |
|------|----------------|-----------------|------|
| **B** | **SBC** | 2.53 (고) | 논문 B센터(고변동)→SBC와 **일치** |
| **E** | **SBC** | 2.57 (고) | 고변동 type → SBC 우세 |
| **A** | **SBC** | 2.27 (중·저) | CV 대비 SBC 우세 — **부분 예외** |
| **C** | ML | **2.58 (최고)** | 최고 CV인데 ML — **예외** (ML 세분화가 유리) |
| **D** | ML | 2.25 (최저) | 저변동 → ML 우세, 논문 A센터 스토리와 **유사** |

**전체: SBC 3 (A,B,E) : ML 2 (C,D)**

#### 논문 방법론과의 정합성
- **CV(변동계수)**는 mean·std와 달리 **규모를 제거한 순수 변동성** 지표입니다. B·C·E는 record CV 기준 **고변동**, D·A는 **상대적 저변동**입니다.
- **B·E에서 SBC(rule-base) 우세** → 수요 패턴 규칙 기반 분류가 **고변동 환경**에서 예측에 도움이 될 수 있음을 시사합니다. 이는 논문에서 Center B(SBC 유리)와 방향이 맞습니다.
- **D(저변동)에서 ML 우세** → 임베딩·ML 클러스터링이 **저변동·대량 판매** type에서 유리할 수 있음 (논문 Center A ↔ ML 스토리).
- **C는 최고 CV이나 ML 우세** — scheme 선택이 CV만으로 결정되지 않음을 보여 주는 **반례**입니다.

#### 한계 (반드시 명시)
> 본 결과는 **Ecuador Favorita 5개 type·165 시계열·3주 검증**에 한정됩니다.  
> SBC와 ML이 **모든 데이터에서 동일한 규칙으로 우세하지는 않습니다.**  
> 다만 **「고변동 → SBC 경향, 저변동 → ML 경향」**이라는 가설이 **완전 일치는 아니지만 부분적으로 지지**되며,  
> **어느 scheme을 쓸지 판단할 때 변동성(CV)을 함께 보는 것**이 타당함을 확인한 것으로 해석합니다.

12장 RIDR·System-Level 분석과 함께 읽으면 type별 변동 구조를 더 자세히 볼 수 있습니다.